# PySpark kết nối HDFS trong JupyterLab – pipeline 5 bước trên cụm Docker

[1. ĐỌC] ➔ [2. KIỂM TRA & LÀM SẠCH] ➔ [3. BIẾN ĐỔI] ➔ [4. TỔNG HỢP] ➔ [5. GHI]. Mục 2–4 dưới đây minh hoạ Bước 1–2 bằng tay; mục 5 chạy cả 5 bước.

Notebook chạy trong container `mis5007-spark-notebook` của cụm (http://localhost:8888). Trước khi chạy, làm trên máy:
1. `bash scripts/chay_toan_bo.sh` (tạo `data/lake/bronze/`; hoặc chạy riêng `scripts/1_thu_thap/1b_chuyen_doi_bronze.py`, `1c_tra_cuu_khong_gian.py`).
2. `bash scripts/cum_hdfs_yarn/hdfs_yarn_cli.sh truoc` (đưa tầng Bronze lên HDFS `/mis5007/bronze` bằng `hdfs dfs -put`).

**Không bịa số:** mọi con số trong báo cáo lấy từ file CSV trong `outputs/tables/`, không chép từ màn hình notebook.

## 1. Mở phiên Spark và trỏ tới HDFS

In [ ]:
import os  # đọc/ghi biến môi trường
os.environ.setdefault('HADOOP_USER_NAME', 'root')  # ghi HDFS bằng user root (cụm của lớp bật kiểm tra quyền)
from pyspark.sql import SparkSession, functions as F, types as T  # thành phần Spark SQL
PROJECT = '/home/jovyan/work/mis5007_tram_y_te'  # thư mục dự án được gắn vào container
HDFS = 'hdfs://namenode:9000/mis5007'  # gốc hồ dữ liệu trên HDFS
spark = (SparkSession.builder.appName('mis5007_notebook').master('local[*]')  # tạo phiên Spark
         .config('spark.driver.memory', '2g').getOrCreate())  # bộ nhớ driver 2 GB
jvm, hconf = spark.sparkContext._jvm, spark.sparkContext._jsc.hadoopConfiguration()  # cầu nối Java để duyệt HDFS

def bronze(nguon, bang):  # đường dẫn bảng Bronze của lần nạp mới nhất
    p = jvm.org.apache.hadoop.fs.Path(f'{HDFS}/bronze/source_system={nguon}')  # thư mục của nguồn
    ngay = sorted(str(s.getPath().getName()) for s in p.getFileSystem(hconf).listStatus(p))  # các thư mục ingestion_date=...
    return f'{HDFS}/bronze/source_system={nguon}/{ngay[-1]}/{bang}'  # lần nạp mới nhất
spark  # hiển thị thông tin phiên

## 2. Đọc Bronze từ HDFS với schema khai báo
Không để Spark tự đoán kiểu. Chế độ `PERMISSIVE` đưa dòng sai kiểu hoặc sai số cột vào cột `_corrupt_record` thay vì làm hỏng cả lần đọc.

In [ ]:
schema = T.StructType([T.StructField('year', T.IntegerType()), T.StructField('row', T.IntegerType()),  # năm, hàng pixel
                       T.StructField('col', T.IntegerType()), T.StructField('pop_raw', T.DoubleType()),  # cột pixel, giá trị thô
                       T.StructField('_corrupt_record', T.StringType())])  # cột hứng dòng hỏng
pop = (spark.read.option('header', True).option('mode', 'PERMISSIVE')  # đọc CSV có tiêu đề, chế độ PERMISSIVE
       .option('columnNameOfCorruptRecord', '_corrupt_record').schema(schema)  # schema khai báo
       .csv(bronze('worldpop', 'pop_100m_raw.csv'))).cache()  # file Bronze mới nhất, giữ trong bộ nhớ
pop.printSchema()  # in schema
print('So dong tho:', pop.count())  # đếm dòng
pop.show(5)  # xem 5 dòng đầu

## 3. Dòng nào sạch, dòng nào bị loại?
`-99999` là mã NoData của WorldPop (pixel không có dân cư) – **không phải lỗi**: pipeline xếp các dòng này vào `silver/excluded_records` (ngoài phạm vi), không vào `silver/quarantine_records`.

In [ ]:
(pop.withColumn('tinh_trang', F.when(F.col('_corrupt_record').isNotNull(), 'sai kieu / sai so cot')  # phân loại từng dòng
                              .when(F.col('pop_raw').isNull(), 'thieu gia tri')  # (tiếp)
                              .when(F.col('pop_raw') < 0, 'gia tri am (NoData)').otherwise('gia tri hop le'))  # (tiếp)
    .groupBy('year', 'tinh_trang').count().orderBy('year', 'tinh_trang').show(30, truncate=False))  # đếm theo năm và loại

## 4. Spark SQL trên dữ liệu đã lọc

In [ ]:
pop.filter('pop_raw >= 0').createOrReplaceTempView('pop_hop_le')  # bảng tạm chỉ gồm giá trị không âm
spark.sql('''SELECT year, COUNT(*) AS so_pixel_co_gia_tri, ROUND(SUM(pop_raw)) AS dan_trong_hop_bao
             FROM pop_hop_le GROUP BY year ORDER BY year''').show()  # tổng dân trong hộp bao theo năm

## 5. Chạy toàn bộ pipeline Spark 5 bước trên HDFS
Script đọc Bronze từ `/mis5007/bronze`, ghi lỗi thật vào `/mis5007/silver/quarantine_records`, dòng ngoài phạm vi vào `/mis5007/silver/excluded_records`, bảng sạch vào `/mis5007/silver`, KPI và dự báo vào `/mis5007/gold`, CSV vào `outputs/tables/`. Dừng phiên Spark của notebook trước để tránh tranh bộ nhớ.

In [ ]:
spark.stop()  # dừng phiên của notebook
!cd {PROJECT} && HDFS_URI={HDFS} python scripts/2_spark_5_buoc/chay_spark.py 2>&1 | grep -E 'INFO|WARNING|ERROR|Traceback'  # chạy pipeline, chỉ in dòng log chính

## 6. Đối soát – mọi dòng phải là ĐẠT
Dòng LỆCH → đọc `logs/spark_5_buoc_*.log`.

In [ ]:
import pandas as pd  # đọc CSV kết quả
pd.set_option('display.max_colwidth', 120)  # hiện chuỗi dài
pd.read_csv(f'{PROJECT}/outputs/tables/7_kiem_chung/doi_soat_spark.csv')  # bảng đối soát

## 6b. Thời gian từng bước của khung 5 bước
Cột `ty_le_thoi_gian_pct` cho biết bước nào chiếm nhiều thời gian nhất.

In [ ]:
pd.read_csv(f'{PROJECT}/outputs/tables/7_kiem_chung/thoi_gian_5_buoc.csv')  # thời gian 5 bước của lần chạy vừa rồi

## 7. Dòng sạch, lỗi dữ liệu (quarantine) và ngoài phạm vi (excluded)

In [ ]:
pd.read_csv(f'{PROJECT}/outputs/tables/2_chat_luong/spark_so_dong_theo_nguon.csv')  # số dòng theo nguồn

In [ ]:
pd.read_csv(f'{PROJECT}/outputs/tables/2_chat_luong/spark_quarantine_tong_hop.csv')  # dòng bị loại theo lý do

## 8. Bảng KPI

In [ ]:
pd.read_csv(f'{PROJECT}/outputs/tables/3_mo_ta_kpi/kpi_tong_hop_spark.csv')  # KPI tổng hợp năm 2024

In [ ]:
pd.read_csv(f'{PROJECT}/outputs/tables/3_mo_ta_kpi/kpi_theo_nam_spark.csv')  # KPI theo năm 2015–2024

In [ ]:
pd.read_csv(f'{PROJECT}/outputs/tables/3_mo_ta_kpi/kpi_theo_nhom_spark.csv')  # KPI theo nhóm địa bàn

## 9. Minh chứng
- HDFS NameNode: http://localhost:9871 → Utilities → Browse the file system → `/mis5007` (chụp `bronze`, `silver`, `gold`).
- YARN: http://localhost:8088.
- Trên máy: `bash scripts/cum_hdfs_yarn/hdfs_yarn_cli.sh sau` ghi kết quả các lệnh `hdfs dfs` và `yarn` vào `logs/`.